# CERT r4.2 — Exploratory Data Analysis

This notebook performs the same EDA checks on all CERT behavioral datasets using a common format.

Datasets covered:
- `logon.csv`
- `file.csv`
- `device.csv`
- `email.csv`
- `http.csv`

The HTTP dataset is handled with chunked reading because it is much larger than the other files.

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 1. Dataset Configuration

In [ ]:
DATASETS = {
    "Logon": "logon.csv",
    "File": "file.csv",
    "Device": "device.csv",
    "Email": "email.csv",
    "HTTP": "http.csv"
}

for name, path in DATASETS.items():
    print(f"{name:10} : {path} | exists = {os.path.exists(path)}")

## 2. Common EDA Function

For each dataset we check:
- first five rows
- number of rows and columns
- column names and data types
- missing values
- duplicate rows
- number of unique values in every column
- whether each column is completely unique

This keeps the EDA format identical across datasets.

In [ ]:
def run_eda(df, dataset_name):
    print("=" * 70)
    print(f"EDA FOR {dataset_name.upper()}")
    print("=" * 70)

    print("\n1. First 5 rows")
    display(df.head())

    print("\n2. Shape")
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    print("\n3. Columns and data types")
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values
    }))

    print("\n4. Missing values")
    missing = df.isna().sum()
    display(pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": (missing / len(df) * 100).round(4)
    }))

    print("\n5. Duplicate rows")
    print(f"Duplicate rows: {df.duplicated().sum():,}")

    print("\n6. Unique values")
    unique_counts = pd.DataFrame({
        "unique_count": df.nunique(dropna=False),
        "is_unique": df.nunique(dropna=False) == len(df)
    })
    display(unique_counts)

    print("\n7. Basic numerical summary")
    display(df.describe(include="all").T)

    print("\n" + "=" * 70)

## 3. EDA for Logon

In [ ]:
df_logon = pd.read_csv(DATASETS["Logon"])
run_eda(df_logon, "Logon")

## 4. EDA for File

In [ ]:
df_file = pd.read_csv(DATASETS["File"])
run_eda(df_file, "File")

## 5. EDA for Device

In [ ]:
df_device = pd.read_csv(DATASETS["Device"])
run_eda(df_device, "Device")

## 6. EDA for Email

In [ ]:
df_email = pd.read_csv(DATASETS["Email"])
run_eda(df_email, "Email")

## 7. EDA for HTTP

`http.csv` is much larger than the other behavioral files. Loading the complete file into memory just for basic EDA is unnecessary and can consume a large amount of RAM.

Therefore, we use:
- chunked reading
- a fixed-size sample for row-level EDA
- a separate row-count pass for the complete file

This lets us inspect the HTTP schema without loading the entire dataset at once.

In [ ]:
HTTP_CHUNK_SIZE = 100_000
HTTP_SAMPLE_SIZE = 150_000

http_sample_parts = []
http_total_rows = 0

for chunk in pd.read_csv(DATASETS["HTTP"], chunksize=HTTP_CHUNK_SIZE):
    http_total_rows += len(chunk)

    remaining = HTTP_SAMPLE_SIZE - sum(len(x) for x in http_sample_parts)
    if remaining > 0:
        http_sample_parts.append(chunk.head(remaining))

df_http_sample = pd.concat(http_sample_parts, ignore_index=True)

print(f"Complete HTTP row count: {http_total_rows:,}")
print(f"HTTP rows used for sample EDA: {len(df_http_sample):,}")

run_eda(df_http_sample, "HTTP SAMPLE")

## 8. AutoViz

In [ ]:
# Install once if AutoViz is not already installed.
# !pip install autoviz

from autoviz.AutoViz_Class import AutoViz_Class

AV = AutoViz_Class()

In [ ]:
def run_autoviz(df, dataset_name, max_rows=150000):
    print(f"Running AutoViz for {dataset_name}...")
    return AV.AutoViz(
        "",
        sep=",",
        depVar="",
        dfte=df,
        header=0,
        verbose=1,
        lowess=False,
        chart_format="svg",
        max_rows_analyzed=min(max_rows, len(df)),
        max_cols_analyzed=30
    )

In [ ]:
autoviz_reports = {}

for name, df in {
    "Logon": df_logon,
    "File": df_file,
    "Device": df_device,
    "Email": df_email,
    "HTTP Sample": df_http_sample
}.items():
    autoviz_reports[name] = run_autoviz(df, name)

## 9. EDA Summary

After this notebook, we should know for every source:
- its schema
- its size
- whether missing values exist
- whether duplicate rows exist
- the cardinality of categorical columns
- which columns are identifiers
- which columns represent user, time, computer, device, file, email or web behavior

These findings will be used in the next stage: **unifying the different event sources into a common behavioral representation**.